# 02 — Bronze → Silver (Delta Lake)

Lê as tabelas **Delta** da camada **Bronze** no MinIO, aplica regras de
**Data Quality** e persiste como **Delta Lake** na camada **Silver**.

### Regras de DQ aplicadas
| Regra | Descrição |
|---|---|
| `drop_duplicates` | Remove linhas completamente duplicadas |
| `drop_critical_nulls` | Remove linhas com nulo em colunas-chave por tabela |
| `fill_optional_nulls` | Preenche nulos em colunas opcionais com padrão de negócio |
| `cast_types` | Garante tipos corretos (timestamps, doubles, ints) |
| `snake_case` | Renomeia colunas para snake_case |

### Requisitos
- MinIO rodando em `localhost:9000` (`docker compose up -d`)
- Tabelas Delta já persistidas no bucket `bronze` (notebook `01a`)
- PySpark + delta-spark instalados no venv

### Papermill
```bash
papermill notebooks/02_bronze_to_silver.ipynb output.ipynb \
    -p bronze_bucket bronze \
    -p silver_bucket silver
```

In [ ]:
from __future__ import annotations

import json
import logging
import os
import re
from pathlib import Path
from typing import Any

from pyspark.sql import DataFrame, SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

In [ ]:
# ──────────────────────────────────────────────
# Logging
# ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("bronze_to_silver")

In [ ]:
# ──────────────────────────────────────────────
# Configuração de DQ por tabela
# ──────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent

# Colunas-chave: linhas com nulo nessas colunas são removidas.
CRITICAL_COLS: dict[str, list[str]] = {
    "olist_customers_dataset":           ["customer_id", "customer_unique_id"],
    "olist_geolocation_dataset":         ["geolocation_zip_code_prefix"],
    "olist_order_items_dataset":         ["order_id", "product_id", "seller_id"],
    "olist_order_payments_dataset":      ["order_id"],
    "olist_order_reviews_dataset":       ["review_id", "order_id"],
    "olist_orders_dataset":              ["order_id", "customer_id", "order_status", "order_purchase_timestamp"],
    "olist_products_dataset":            ["product_id"],
    "olist_sellers_dataset":             ["seller_id"],
    "product_category_name_translation": ["product_category_name"],
    "agents":                            ["agent_id", "alias", "team"],
    "support_tickets":                   ["ticket_id", "order_id", "customer_id", "agent_id", "status", "opened_at"],
    "support_ticket_messages":           ["message_id", "ticket_id", "sender", "body"],
}

# Colunas opcionais: nulos são preenchidos com um valor padrão de negócio.
OPTIONAL_FILL: dict[str, dict[str, Any]] = {
    "olist_order_payments_dataset": {
        "payment_installments": 1,
        "payment_value": 0.0,
    },
    "olist_order_reviews_dataset": {
        "review_comment_title": "",
        "review_comment_message": "",
    },
    "olist_products_dataset": {
        "product_category_name": "unknown",
        "product_name_lenght": 0,
        "product_description_lenght": 0,
        "product_photos_qty": 0,
    },
    "support_tickets": {
        "channel": "unknown",
        "priority": "normal",
    },
}

# Casts explícitos de tipo por coluna (por tabela)
TYPE_CASTS: dict[str, dict[str, T.DataType]] = {
    "olist_order_items_dataset": {
        "price": T.DoubleType(),
        "freight_value": T.DoubleType(),
        "shipping_limit_date": T.TimestampType(),
    },
    "olist_order_payments_dataset": {
        "payment_value": T.DoubleType(),
        "payment_installments": T.IntegerType(),
        "payment_sequential": T.IntegerType(),
    },
    "olist_orders_dataset": {
        "order_purchase_timestamp": T.TimestampType(),
        "order_approved_at": T.TimestampType(),
        "order_delivered_carrier_date": T.TimestampType(),
        "order_delivered_customer_date": T.TimestampType(),
        "order_estimated_delivery_date": T.TimestampType(),
    },
    "olist_order_reviews_dataset": {
        "review_score": T.IntegerType(),
        "review_creation_date": T.TimestampType(),
        "review_answer_timestamp": T.TimestampType(),
    },
    "olist_products_dataset": {
        "product_weight_g": T.DoubleType(),
        "product_length_cm": T.DoubleType(),
        "product_height_cm": T.DoubleType(),
        "product_width_cm": T.DoubleType(),
    },
    "olist_geolocation_dataset": {
        "geolocation_lat": T.DoubleType(),
        "geolocation_lng": T.DoubleType(),
        "geolocation_zip_code_prefix": T.IntegerType(),
    },
    "support_tickets": {
        "opened_at": T.TimestampType(),
        "closed_at": T.TimestampType(),
        "first_response_minutes": T.IntegerType(),
        "sla_target_hours": T.IntegerType(),
        "resolution_hours": T.IntegerType(),
        "requires_seller_action": T.BooleanType(),
    },
    "support_ticket_messages": {
        "sent_at": T.TimestampType(),
    },
}

# Tabelas a processar (Bronze → Silver)
TABLES_TO_PROCESS = list(CRITICAL_COLS.keys())

In [ ]:
# ──────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────
def _load_env() -> None:
    """Carrega variáveis do arquivo .env (formato KEY=VALUE) sem depender de python-dotenv."""
    env_path = PROJECT_ROOT / ".env"
    if not env_path.exists():
        log.warning(".env não encontrado em %s – usando variáveis do sistema", env_path)
        return
    with open(env_path) as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

_load_env()

In [ ]:
# ──────────────────────────────────────────────
# Parâmetros (Papermill)
# ──────────────────────────────────────────────
# Esta célula é marcada com a tag 'parameters'.
# O Papermill injeta uma nova célula logo abaixo com os valores recebidos.
bronze_bucket: str = os.getenv("MINIO_BUCKET_BRONZE", "bronze")
silver_bucket: str = os.getenv("MINIO_BUCKET_SILVER", "silver")
write_mode: str = "overwrite"   # 'overwrite' | 'append'

In [ ]:
# ──────────────────────────────────────────────
# SparkSession
# ──────────────────────────────────────────────
def _create_spark_session() -> SparkSession:
    """
    Cria uma SparkSession local configurada para:
      - Acessar MinIO via protocolo S3A (hadoop-aws)
      - Usar Delta Lake como formato de tabela
    """
    endpoint   = os.getenv("MINIO_ENDPOINT",   "http://localhost:9000")
    access_key = os.getenv("MINIO_ACCESS_KEY", "minioadmin")
    secret_key = os.getenv("MINIO_SECRET_KEY", "minioadmin")

    builder = (
        SparkSession.builder
        .appName("BronzeToSilver")
        .master("local[*]")
        # ── Delta Lake ──
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        # ── Hadoop / S3A para MinIO ──
        .config("spark.hadoop.fs.s3a.endpoint", endpoint)
        .config("spark.hadoop.fs.s3a.access.key", access_key)
        .config("spark.hadoop.fs.s3a.secret.key", secret_key)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
        )
        # ── Performance ──
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.driver.memory", "2g")
    )

    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    return spark

spark = _create_spark_session()

In [ ]:
# ──────────────────────────────────────────────
# Funções de DQ
# ──────────────────────────────────────────────
def _to_snake_case(name: str) -> str:
    """Converte um nome de coluna para snake_case."""
    name = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", name)
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"[\s\-]+", "_", name)
    return name.lower()


def apply_snake_case(df: DataFrame) -> tuple[DataFrame, int]:
    """Renomeia todas as colunas para snake_case. Retorna (df, colunas_renomeadas)."""
    renamed = 0
    for col in df.columns:
        new_name = _to_snake_case(col)
        if new_name != col:
            df = df.withColumnRenamed(col, new_name)
            renamed += 1
    return df, renamed


def apply_dedup(df: DataFrame) -> tuple[DataFrame, int]:
    """Remove duplicatas completas. Retorna (df, linhas_removidas)."""
    before = df.count()
    df = df.dropDuplicates()
    return df, before - df.count()


def apply_critical_null_drop(
    df: DataFrame, critical_cols: list[str]
) -> tuple[DataFrame, int]:
    """
    Remove linhas com nulo em colunas-chave.
    Apenas colunas que existem no DataFrame são verificadas
    (evita erro em tabelas que não possuem a coluna).
    Retorna (df, linhas_removidas).
    """
    cols_present = [c for c in critical_cols if c in df.columns]
    if not cols_present:
        return df, 0
    before = df.count()
    df = df.dropna(subset=cols_present)
    return df, before - df.count()


def apply_optional_fill(
    df: DataFrame, fill_map: dict[str, Any]
) -> tuple[DataFrame, int]:
    """
    Preenche nulos em colunas opcionais com valores-padrão.
    Apenas colunas presentes no DataFrame são preenchidas.
    Retorna (df, colunas_preenchidas).
    """
    valid_map = {k: v for k, v in fill_map.items() if k in df.columns}
    if not valid_map:
        return df, 0
    df = df.fillna(valid_map)
    return df, len(valid_map)


def apply_type_casts(
    df: DataFrame, cast_map: dict[str, T.DataType]
) -> tuple[DataFrame, int]:
    """
    Aplica casts explícitos de tipo.
    Apenas colunas presentes no DataFrame são castadas.
    Retorna (df, colunas_castadas).
    """
    cast_count = 0
    for col_name, dtype in cast_map.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, F.col(col_name).cast(dtype))
            cast_count += 1
    return df, cast_count


def count_remaining_nulls(df: DataFrame) -> dict[str, int]:
    """
    Conta nulos restantes por coluna após todas as transformações.
    Colunas sem nulos são omitidas do resultado.
    """
    # Usa uma única passagem agregada para eficiência
    null_counts = df.select(
        [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
    ).collect()[0].asDict()
    return {k: v for k, v in null_counts.items() if v > 0}

In [ ]:
# ──────────────────────────────────────────────
# Pipeline principal por tabela
# ──────────────────────────────────────────────
def _process_table(
    spark: SparkSession,
    table_name: str,
    bronze_bucket: str,
    silver_bucket: str,
    write_mode: str,
) -> dict[str, Any]:
    """
    Executa o pipeline Bronze → Silver para uma tabela:
      1. Lê o Delta do Bronze
      2. Aplica snake_case nas colunas
      3. Remove duplicatas
      4. Remove linhas com nulos críticos
      5. Preenche nulos opcionais
      6. Aplica casts de tipo
      7. Adiciona metadados Silver
      8. Persiste como Delta no Silver

    Retorna o log de DQ da tabela.
    """
    source_path = f"s3a://{bronze_bucket}/{table_name}"
    target_path = f"s3a://{silver_bucket}/{table_name}"

    log.info("[%s] Lendo Bronze: %s", table_name, source_path)
    df = spark.read.format("delta").load(source_path)

    rows_bronze = df.count()
    log.info("[%s] Linhas no Bronze: %d", table_name, rows_bronze)

    # 1 ── snake_case
    df, cols_renamed = apply_snake_case(df)

    # 2 ── Deduplica
    df, dropped_dupes = apply_dedup(df)
    log.info("[%s] Duplicatas removidas: %d", table_name, dropped_dupes)

    # 3 ── Drop nulos críticos
    critical_cols = CRITICAL_COLS.get(table_name, [])
    df, dropped_nulls = apply_critical_null_drop(df, critical_cols)
    log.info("[%s] Linhas removidas por nulo crítico: %d", table_name, dropped_nulls)

    # 4 ── Fill nulos opcionais
    fill_map = OPTIONAL_FILL.get(table_name, {})
    df, cols_filled = apply_optional_fill(df, fill_map)
    if cols_filled > 0:
        log.info("[%s] Colunas opcionais preenchidas: %d", table_name, cols_filled)

    # 5 ── Casts de tipo
    cast_map = TYPE_CASTS.get(table_name, {})
    df, cols_cast = apply_type_casts(df, cast_map)
    if cols_cast > 0:
        log.info("[%s] Colunas castadas: %d", table_name, cols_cast)

    # 6 ── Metadados Silver
    df = (
        df
        .withColumn("_silver_timestamp", F.current_timestamp())
        .withColumn("_silver_source", F.lit(f"{bronze_bucket}/{table_name}"))
    )

    # 7 ── Contagem de nulos restantes (antes de escrever)
    nulls_remaining = count_remaining_nulls(df)

    rows_silver = df.count()
    log.info("[%s] Linhas no Silver: %d", table_name, rows_silver)

    # 8 ── Persiste
    log.info("[%s] Persistindo Delta em: %s", table_name, target_path)
    (
        df.write
        .format("delta")
        .mode(write_mode)
        .option("overwriteSchema", "true")
        .save(target_path)
    )

    return {
        "table": table_name,
        "rows_bronze": rows_bronze,
        "rows_silver": rows_silver,
        "dropped_duplicates": dropped_dupes,
        "dropped_critical_nulls": dropped_nulls,
        "optional_cols_filled": cols_filled,
        "cols_cast": cols_cast,
        "cols_renamed_to_snake_case": cols_renamed,
        "nulls_remaining_by_col": nulls_remaining,
        "status": "ok",
    }

In [ ]:
# ──────────────────────────────────────────────
# Execução: Bronze → Silver
# ──────────────────────────────────────────────
log.info("=" * 60)
log.info("Iniciando pipeline Bronze → Silver")
log.info("=" * 60)
log.info("  Bronze bucket: s3a://%s/", bronze_bucket)
log.info("  Silver bucket: s3a://%s/", silver_bucket)

dq_log: list[dict[str, Any]] = []
success_count = 0

for table_name in TABLES_TO_PROCESS:
    try:
        result = _process_table(
            spark=spark,
            table_name=table_name,
            bronze_bucket=bronze_bucket,
            silver_bucket=silver_bucket,
            write_mode=write_mode,
        )
        dq_log.append(result)
        success_count += 1
    except Exception as exc:
        log.error("[%s] FALHA: %s", table_name, exc)
        dq_log.append({
            "table": table_name,
            "status": "error",
            "error": str(exc),
        })

log.info("-" * 60)
log.info(
    "Pipeline concluído: %d/%d tabelas processadas",
    success_count,
    len(TABLES_TO_PROCESS),
)

In [ ]:
# ──────────────────────────────────────────────
# Log de DQ — resumo estruturado
# ──────────────────────────────────────────────
from datetime import datetime

log.info("=" * 60)
log.info("LOG DE DATA QUALITY")
log.info("=" * 60)

for entry in dq_log:
    if entry["status"] == "error":
        log.error(
            "  ✘ %-45s | ERRO: %s",
            entry["table"],
            entry.get("error"),
        )
        continue

    retention_pct = (
        round(entry["rows_silver"] / entry["rows_bronze"] * 100, 2)
        if entry["rows_bronze"] > 0
        else 0.0
    )
    log.info(
        "  ✔ %-45s | bronze=%d  silver=%d  retention=%.1f%%  "
        "dupes=%d  crit_nulls=%d  nulls_remaining=%s",
        entry["table"],
        entry["rows_bronze"],
        entry["rows_silver"],
        retention_pct,
        entry["dropped_duplicates"],
        entry["dropped_critical_nulls"],
        entry["nulls_remaining_by_col"] or "{}",
    )

# Exibe o log completo como JSON no output do notebook (útil para pipelines)
dq_log_json = json.dumps(dq_log, ensure_ascii=False, indent=2, default=str)
print(dq_log_json)

# Persiste o log de DQ em arquivo para auditoria
dq_log_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
dq_log_path = f"/tmp/dq_log_bronze_to_silver_{dq_log_timestamp}.json"
Path(dq_log_path).write_text(dq_log_json, encoding="utf-8")
log.info("DQ log persistido em: %s", dq_log_path)

In [ ]:
# ──────────────────────────────────────────────
# Verificação das tabelas Silver (Delta history)
# ──────────────────────────────────────────────
log.info("=" * 60)
log.info("VERIFICAÇÃO DAS TABELAS DELTA SILVER")
log.info("=" * 60)

for table_name in TABLES_TO_PROCESS:
    target_path = f"s3a://{silver_bucket}/{table_name}"
    try:
        dt = DeltaTable.forPath(spark, target_path)
        history = dt.history(1).select("version", "timestamp", "operation").collect()
        row = history[0]
        log.info(
            "  ✔ %-45s | v%s | %s | %s",
            table_name,
            row["version"],
            row["timestamp"],
            row["operation"],
        )
    except Exception as exc:
        log.error("  ✘ %-45s | ERRO: %s", table_name, exc)

In [ ]:
# ──────────────────────────────────────────────
# Finalização
# ──────────────────────────────────────────────
spark.stop()

failed_tables = [e["table"] for e in dq_log if e["status"] == "error"]
if failed_tables:
    raise RuntimeError(
        f"Falha no processamento de {len(failed_tables)} tabela(s): {failed_tables}"
    )

log.info("✔ Pipeline Bronze → Silver finalizado com sucesso!")